In [0]:
from utils.helper import BatchControlHelper
from pyspark.sql import DataFrame
from pyspark.sql import functions as f
import json
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("environment", "dev")

dbutils.widgets.text("processing_date", "2024-12-31", "As-of date used when generating the initial load (YYYY-MM-DD)")
dbutils.widgets.dropdown("force_reload", "false", ["false", "true"], "Reprocess tables even if already SUCCESS")

processing_date = dbutils.widgets.get("processing_date")
force_reload = dbutils.widgets.get("force_reload") == "true"
env = dbutils.widgets.get("environment")

batch_id = f"ods-reports-{processing_date.replace('-', '')}"


## 1. Current transaction operations view


In [0]:
transaction = spark.sql(f"SELECT * FROM {env}_banking.ods.transaction")
transaction = (transaction.withColumn('row_effective_at', f.col('source_modified_at'))
    .withColumn('event_arrival_lag_days', f.when(
        f.col('arrival_timestamp').isNotNull() & f.col('transaction_timestamp').isNotNull(),
        f.greatest(f.lit(0), f.datediff(f.to_date('arrival_timestamp'), f.col('transaction_timestamp')))))
    .withColumn('is_late_arrival',
        f.coalesce(f.col('is_late_arrival').cast('boolean'), f.lit(False)) |
        (f.col('arrival_timestamp').isNotNull() & f.col('transaction_timestamp').isNotNull() &
         (f.to_date('arrival_timestamp') > f.col('transaction_timestamp'))))
)

account_context = spark.table(f"SELECT * FROM {env}_banking.ods.account").alias('a')
branch_context = spark.table(f"SELECT * FROM {env}_banking.ods.branch").select('branch_id', 'branch_name', 'region').alias('b')
customer_today = (spark.table(f"SELECT * FROM {env}_banking.ods.customer_history")
    .filter((f.col('row_effective_date') <= f.current_date()) &
            (f.col('row_end_date').isNull() | (f.current_date() < f.col('row_end_date'))))
    .select('customer_id', 'customer_segment', 'kyc_status'))

transaction_operational = (transaction.alias('t')
    .join(account_context, f.col('t.account_id') == f.col('a.account_id'), 'left')
    .join(branch_context, f.col('a.branch_id') == f.col('b.branch_id'), 'left')
    .join(customer_today.alias('c'), f.col('a.primary_customer_id') == f.col('c.customer_id'), 'left')
    .select(f.col('t.transaction_hk'), f.col('t.transaction_id'),
        f.col('t.account_id'), f.col('a.primary_customer_id'), f.col('c.customer_segment'), f.col('c.kyc_status'),
        f.col('a.account_type'), f.col('a.account_status'), f.col('a.branch_id'), f.col('b.branch_name'), f.col('b.region'),
        f.col('t.transaction_effective_at'), f.col('t.transaction_effective_date'),
        f.col('t.transaction_type'), f.col('t.transaction_amount').cast('decimal(18,2)').alias('transaction_amount'),
        f.col('t.currency_code'), f.col('t.transaction_status'), f.col('t.merchant_name'), f.col('t.merchant_category'),
        f.col('t.arrival_at'), f.to_date('t.arrival_at').alias('arrival_date'), f.col('t.event_arrival_lag_days'),
        f.col('t.is_late_arrival'), f.col('t.row_effective_at'), f.col('t.source_system'), f.col('t.batch_id'),
        f.col('t.record_source'), f.col('t.ingested_at'), f.col('t.row_hash')))


In [0]:
transaction_operational.show(30)

## 2. Account operating position


In [0]:
balance_window = Window.partitionBy('account_id').orderBy(
    f.col('balance_date').desc(), f.col('source_modified_at').desc_nulls_last())
latest_balance = (spark.table(ods_table('ods_account_daily_balance'))
    .filter(f.to_date('balance_date') <= f.current_date())
    .withColumn('_balance_rank', f.row_number().over(balance_window)).filter(f.col('_balance_rank') == 1)
    .select('account_id', f.to_date('balance_date').alias('balance_as_of_date'),
        f.col('available_balance_amount').cast('decimal(18,2)').alias('available_balance_amount'),
        f.col('ledger_balance_amount').cast('decimal(18,2)').alias('ledger_balance_amount'),
        f.col('overdraft_limit_amount').cast('decimal(18,2)').alias('overdraft_limit_amount')))

account_txn = spark.table(ods_table('ods_transaction_operational_current'))
activity_30d = (account_txn
    .filter(f.col('transaction_effective_date').between(f.date_sub(f.current_date(), 29), f.current_date()))
    .groupBy('account_id')
    .agg(f.count('*').alias('transactions_last_30d'),
        f.sum(f.when(f.col('transaction_status') == 'POSTED', 1).otherwise(0)).alias('posted_transactions_last_30d'),
        f.sum(f.when((f.col('transaction_status') == 'POSTED') & (f.col('transaction_amount') > 0), f.col('transaction_amount'))
              .otherwise(f.lit(0).cast('decimal(18,2)'))).alias('credits_last_30d'),
        f.sum(f.when((f.col('transaction_status') == 'POSTED') & (f.col('transaction_amount') < 0), -f.col('transaction_amount'))
              .otherwise(f.lit(0).cast('decimal(18,2)'))).alias('debits_last_30d'),
        f.sum(f.when(f.col('is_late_arrival'), 1).otherwise(0)).alias('late_arrivals_last_30d')))
last_activity_window = Window.partitionBy('account_id').orderBy(
    f.col('transaction_effective_at').desc_nulls_last(), f.col('transaction_id').desc())
last_activity = (account_txn.withColumn('_activity_rank', f.row_number().over(last_activity_window))
    .filter(f.col('_activity_rank') == 1)
    .select('account_id', f.col('transaction_effective_at').alias('last_transaction_at'),
        f.col('transaction_status').alias('last_transaction_status')))

account_operational = (spark.table(ods_table('ods_account')).alias('a')
    .join(spark.table(ods_table('ods_branch')).select('branch_id', 'branch_name', 'region').alias('b'),
        f.col('a.branch_id') == f.col('b.branch_id'), 'left')
    .join(customer_today.alias('c'), f.col('a.primary_customer_id') == f.col('c.customer_id'), 'left')
    .join(latest_balance.alias('bal'), f.col('a.account_id') == f.col('bal.account_id'), 'left')
    .join(activity_30d.alias('act'), f.col('a.account_id') == f.col('act.account_id'), 'left')
    .join(last_activity.alias('last'), f.col('a.account_id') == f.col('last.account_id'), 'left')
    .select(f.col('a.account_id'), f.col('a.primary_customer_id'), f.col('c.customer_segment'), f.col('c.kyc_status'),
        f.col('a.branch_id'), f.col('b.branch_name'), f.col('b.region'), f.col('a.account_type'), f.col('a.account_status'),
        f.col('a.open_date'), f.col('a.close_date'), f.col('a.currency_code'), f.col('bal.balance_as_of_date'),
        f.col('bal.available_balance_amount'), f.col('bal.ledger_balance_amount'), f.col('bal.overdraft_limit_amount'),
        f.coalesce(f.col('act.transactions_last_30d'), f.lit(0)).alias('transactions_last_30d'),
        f.coalesce(f.col('act.posted_transactions_last_30d'), f.lit(0)).alias('posted_transactions_last_30d'),
        f.coalesce(f.col('act.credits_last_30d'), f.lit(0).cast('decimal(18,2)')).alias('credits_last_30d'),
        f.coalesce(f.col('act.debits_last_30d'), f.lit(0).cast('decimal(18,2)')).alias('debits_last_30d'),
        f.coalesce(f.col('act.late_arrivals_last_30d'), f.lit(0)).alias('late_arrivals_last_30d'),
        f.col('last.last_transaction_at'), f.col('last.last_transaction_status'),
        f.current_timestamp().alias('read_model_refreshed_at')))
replace_delta_table(account_operational, 'ods_account_operational_current')

## 3. Event-date reporting metrics, line detail and validation


In [0]:
operational_txns = spark.table(ods_table('ods_transaction_operational_current'))
daily_metrics = (operational_txns
    .groupBy('transaction_effective_date', 'branch_id', 'transaction_type', 'transaction_status', 'currency_code')
    .agg(f.count('*').alias('transaction_count'), f.countDistinct('account_id').alias('active_account_count'),
        f.sum(f.when(f.col('transaction_status') == 'POSTED', 1).otherwise(0)).alias('posted_transaction_count'),
        f.sum(f.when(f.col('transaction_amount') > 0, f.col('transaction_amount'))
              .otherwise(f.lit(0).cast('decimal(18,2)'))).alias('credit_amount'),
        f.sum(f.when(f.col('transaction_amount') < 0, -f.col('transaction_amount'))
              .otherwise(f.lit(0).cast('decimal(18,2)'))).alias('debit_amount'),
        f.sum('transaction_amount').alias('net_amount'),
        f.sum(f.when((f.col('transaction_status') == 'POSTED') & (f.col('transaction_amount') > 0), f.col('transaction_amount'))
              .otherwise(f.lit(0).cast('decimal(18,2)'))).alias('posted_credit_amount'),
        f.sum(f.when((f.col('transaction_status') == 'POSTED') & (f.col('transaction_amount') < 0), -f.col('transaction_amount'))
              .otherwise(f.lit(0).cast('decimal(18,2)'))).alias('posted_debit_amount'),
        f.sum(f.when(f.col('transaction_status') == 'POSTED', f.col('transaction_amount'))
              .otherwise(f.lit(0).cast('decimal(18,2)'))).alias('posted_net_amount'),
        f.sum(f.when(f.col('is_late_arrival'), 1).otherwise(0)).alias('late_arrival_count'),
        f.sum(f.when(f.col('is_late_arrival') & (f.col('transaction_status') == 'POSTED'), f.col('transaction_amount'))
              .otherwise(f.lit(0).cast('decimal(18,2)'))).alias('late_arrival_posted_net_amount'),
        f.max('arrival_at').alias('latest_arrival_at'))
    .withColumnRenamed('transaction_effective_date', 'business_date')
    .withColumn('metrics_refreshed_at', f.current_timestamp()))
replace_delta_table(daily_metrics, 'ods_operational_daily_metrics')

line_operational = (spark.table(ods_table('ods_transaction_line')).alias('l')
    .join(operational_txns.select('transaction_id', 'transaction_effective_at', 'transaction_effective_date',
        'account_id', 'branch_id', 'transaction_status', 'is_late_arrival').alias('t'),
        f.col('l.transaction_id') == f.col('t.transaction_id'), 'left')
    .join(spark.table(ods_table('ods_product')).select('product_id', 'product_name', 'product_category').alias('p'),
        f.col('l.product_id') == f.col('p.product_id'), 'left')
    .select(f.col('l.transaction_id'), f.col('l.line_number'), f.col('l.product_id'),
        f.col('p.product_name'), f.col('p.product_category'), f.col('l.quantity'), f.col('l.line_amount'),
        f.col('t.account_id'), f.col('t.branch_id'), f.col('t.transaction_effective_at'),
        f.col('t.transaction_effective_date'), f.col('t.transaction_status'), f.col('t.is_late_arrival'),
        f.col('l.batch_id'), f.col('l.record_source'), f.col('l.ingested_at')))
replace_delta_table(line_operational, 'ods_transaction_line_operational_current')

duplicate_current_count = (operational_txns.groupBy('transaction_id').count()
    .filter(f.col('count') > 1).count())
if duplicate_current_count:
    raise RuntimeError(f'Current ODS transaction view has {duplicate_current_count} duplicate transaction IDs.')
invalid_customer_ranges = (spark.table(ods_table('ods_customer_operational_version'))
    .filter(f.col('row_end_date').isNotNull() & (f.col('row_end_date') <= f.col('row_effective_date'))).count())
if invalid_customer_ranges:
    raise RuntimeError(f'ods_customer_operational_version contains {invalid_customer_ranges} invalid effective-date ranges.')

display(spark.table(ods_table('ods_operational_daily_metrics')).orderBy(f.col('business_date').desc()).limit(100))
display(spark.table(ods_table('ods_late_arrival_monitor')).orderBy(f.col('transaction_effective_date').asc()).limit(100))
display(spark.table(ods_table('ods_account_operational_current')).orderBy(f.col('late_arrivals_last_30d').desc()).limit(100))
print('ODS reporting models passed Silver-grain, current-key and temporal-range checks.')